# schema

> The audio-transcript layer schema (where-graph-begins locked layer schema): `Source -> AudioSegment(coarse boundary) -> AudioRendition(model-input: raw | vocals | …) -> Transcript(per-transcriber variants)` emitted by transcription, extended with the fine `Segment` spine (per-rendition) by decomposition. Deterministic identity tuples per the stage-5 ratified rule; the `AudioRendition` node lets raw + preprocessed model-inputs of one boundary coexist in one graph. `Document` from the pre-CR-18 era dissolves into `Source`.

In [ ]:
#| default_exp schema

In [ ]:
#| export
from pathlib import Path
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional

from cjm_context_graph_layer.identity import derive_node_id
from cjm_context_graph_layer.grammar import OverlayRelations, attribution, make_edge
from cjm_context_graph_primitives.locators import FileRef, GraphNodeRef
from cjm_context_graph_primitives.slices import TimeSlice, CharSlice, FullContent
from cjm_context_graph_primitives.provenance import SourceRef

In [ ]:
#| export
class TranscriptGraphLabels:
    """Node labels of the audio-transcript layer schema."""
    SOURCE = "Source"                  # Provenance root: one ingested media file
    AUDIO_SEGMENT = "AudioSegment"     # Coarse ~5-min spine member: a boundary range of the Source (an audio FACT; rendition-independent)
    AUDIO_RENDITION = "AudioRendition" # A model-input rendition OF an AudioSegment (raw convert | vocals | future enhancement); OWNS the model-input WAV
    TRANSCRIPT = "Transcript"          # Per-transcriber text variant of one AudioRendition
    SEGMENT = "Segment"                # Fine spine member: one VAD chunk (immutable audio + correctable text), per-rendition

    @classmethod
    def all(cls) -> list:  # All schema labels
        """All schema labels."""
        return [cls.SOURCE, cls.AUDIO_SEGMENT, cls.AUDIO_RENDITION, cls.TRANSCRIPT, cls.SEGMENT]

In [ ]:
#| export
def source_node_id(
    content_hash: str,  # Content hash of the source media file ("algo:hexdigest")
) -> str:  # Deterministic Source node id
    """Source identity = the ingested file's content hash (Thread-1 ingested-root identity)."""
    return derive_node_id("source", content_hash)

In [ ]:
#| export
def audio_segment_node_id(
    source_id: str,  # Owning Source node id
    start: float,    # Boundary start (source-coordinate seconds)
    end: float,      # Boundary end (source-coordinate seconds)
) -> str:  # Deterministic AudioSegment node id
    """AudioSegment identity = (source, boundary range).

    Rendition-independent by design: an AudioSegment is a BOUNDARY of the source
    (an audio fact), shared across every model-input rendition of it. The
    boundary computation is pure and deterministic, so re-derivation reproduces
    the id; the model-input WAV lives on the `AudioRendition` child, not here."""
    return derive_node_id("audio-segment", source_id, start, end)

In [ ]:
#| export
def audio_rendition_node_id(
    audio_segment_id: str,  # Owning AudioSegment node id
    chain: List[str],       # Ordered preprocessing-chain descriptors; [] = the raw convert-only rendition
) -> str:  # Deterministic AudioRendition node id
    """AudioRendition identity = (audio segment, preprocessing chain).

    A rendition is one model-input audio OF an AudioSegment. The chain is the
    ordered list of preprocessing steps that produced it (each an opaque
    descriptor string, e.g. ``"source_separation:cjm-media-plugin-demucs@<cfg>"``);
    an EMPTY chain is the raw convert-only rendition, whose id is therefore
    stable and chain-free. The universal 16k-mono resample is implicit (not a
    chain step) — so raw + vocals are distinct renditions under one shared
    AudioSegment, and the layer's identity-mismatch check never collides them.
    Re-derivable from the manifest chain alone (extenders recompute, never
    search)."""
    return derive_node_id("audio-rendition", audio_segment_id, *chain)

In [ ]:
#| export
def transcript_node_id(
    rendition_id: str,  # Owning AudioRendition node id
    transcriber: str,   # Transcriber capability name (e.g. "cjm-transcription-plugin-whisper")
    config_hash: str,   # Transcriber config hash
) -> str:  # Deterministic Transcript node id
    """Transcript identity = (audio rendition, transcriber, config) — MIRRORS the
    capability cache key UNIQUE(audio_path, config_hash), so the graph node is
    the durable face of the cached row (the structural E13 fix). Keyed on the
    RENDITION, so a raw vs vocals transcript of the same boundary are distinct."""
    return derive_node_id("transcript", rendition_id, transcriber, config_hash)

In [ ]:
#| export
def segment_node_id(
    rendition_id: str,     # Owning AudioRendition node id
    vad_config_hash: str,  # VAD capability config hash (skeleton identity input)
    chunk_start: float,    # VAD chunk start (chunk-local seconds within the rendition WAV)
    chunk_end: float,      # VAD chunk end (chunk-local seconds)
) -> str:  # Deterministic Segment node id
    """Fine Segment identity = audio-side only (audio rendition, VAD config, chunk
    range) — so the skeleton's ids are SHARED across transcribers by
    construction (C4 "store agreement once" falls out of identity design). Keyed
    on the RENDITION: each rendition has its own fine spine (vocals isolation can
    yield different VAD chunk boundaries than raw)."""
    return derive_node_id("segment", rendition_id, vad_config_hash, chunk_start, chunk_end)

In [ ]:
#| export
@dataclass
class SourceNode:
    """The provenance root: one ingested media file."""
    content_hash: str            # Content hash over the file ("algo:hexdigest"; the identity input)
    path: str                    # Original media path (location, may dangle; identity is the hash)
    title: Optional[str] = None  # Display title; None = path stem
    media_type: str = "audio"    # Media type

    @property
    def id(self) -> str:  # Deterministic node id
        """Deterministic node id."""
        return source_node_id(self.content_hash)

    def to_graph_node(self) -> Dict[str, Any]:  # Node wire dict
        """Build the Source node wire dict (root_kind=ingested; FileRef provenance)."""
        return {
            "id": self.id,
            "label": TranscriptGraphLabels.SOURCE,
            "properties": {
                "title": self.title or Path(self.path).stem,
                "media_type": self.media_type,
                "path": self.path,
                "root_kind": "ingested",
            },
            "sources": [SourceRef(locator=FileRef(path=self.path),
                                  content_hash=self.content_hash).to_dict()],
        }

In [ ]:
#| export
@dataclass
class AudioSegmentNode:
    """Coarse ~5-min spine member: a BOUNDARY range of the Source (an audio fact),
    shared across every model-input rendition of it.

    Hashless by design: a boundary is not a materialized artifact — a hash over
    the sliced ORIGINAL source bytes is not materializable without extracting
    per-range artifacts (decoded ranges are not byte ranges). The materialized
    model-input WAV lives on the `AudioRendition` child (which carries its
    content hash); the structural chain to the Source rides the PART_OF edge +
    the start/end properties. `segment_path` (the source-codec cut) stays here as
    a rendition-independent archival pointer."""
    source: str                       # Owning Source node id
    index: int                        # Position in the coarse spine (0-based)
    start: float                      # Boundary start (source-coordinate seconds)
    end: float                        # Boundary end (source-coordinate seconds)
    segment_path: Optional[str] = None  # Source-codec cut file (rendition-independent archival pointer)

    @property
    def id(self) -> str:  # Deterministic node id
        """Deterministic node id."""
        return audio_segment_node_id(self.source, self.start, self.end)

    def to_graph_node(self) -> Dict[str, Any]:  # Node wire dict
        """Build the AudioSegment node wire dict (hashless boundary; sources empty)."""
        props: Dict[str, Any] = {
            "index": self.index,
            "start": self.start,
            "end": self.end,
            "source_id": self.source,
        }
        if self.segment_path:
            props["segment_path"] = self.segment_path
        return {
            "id": self.id,
            "label": TranscriptGraphLabels.AUDIO_SEGMENT,
            "properties": props,
            "sources": [],
        }

In [ ]:
#| export
@dataclass
class AudioRenditionNode:
    """A model-input rendition OF an AudioSegment — the materialized 16k-mono WAV
    consumed by transcription/VAD/FA. OWNS the model-input (E14: the audio of
    record); the AudioSegment above it is a hashless boundary.

    Identity = (audio segment, preprocessing chain). The raw convert-only path is
    an empty chain (its id is stable + chain-free); vocals isolation / future
    speech-enhancement are non-empty chains. Raw + vocals are therefore distinct
    renditions that COEXIST under one AudioSegment — the divergent model-input
    hash that used to collide the AudioSegment now lives on distinct rendition
    nodes. Connects UP to its AudioSegment by DERIVED_FROM (it is derived from the
    segment's audio); the production act is recorded separately by a CR-18
    `Derivation` event."""
    audio_segment: str                # Owning AudioSegment node id
    model_input_path: str             # The 16kHz mono WAV consumed by transcription/VAD/FA
    model_input_hash: str             # Content hash over that WAV (the rendition's identity-of-record artifact)
    chain: List[str] = field(default_factory=list)  # Ordered preprocessing-chain descriptors; [] = raw convert-only

    @property
    def id(self) -> str:  # Deterministic node id
        """Deterministic node id."""
        return audio_rendition_node_id(self.audio_segment, self.chain)

    def to_graph_node(self) -> Dict[str, Any]:  # Node wire dict
        """Build the AudioRendition node wire dict (owns the model-input SourceRef)."""
        props: Dict[str, Any] = {
            "audio_segment_id": self.audio_segment,
            "model_input_path": self.model_input_path,
            "chain": list(self.chain),
            "is_raw": not self.chain,
        }
        if self.chain:
            props["preprocessing"] = " | ".join(self.chain)  # readable display label (the retired interim property)
        return {
            "id": self.id,
            "label": TranscriptGraphLabels.AUDIO_RENDITION,
            "properties": props,
            "sources": [SourceRef(locator=FileRef(path=self.model_input_path),
                                  content_hash=self.model_input_hash).to_dict()],
        }

    def derived_edge(self) -> Dict[str, Any]:  # Edge wire dict
        """DERIVED_FROM edge: this rendition derives from its AudioSegment's audio."""
        return make_edge(self.id, self.audio_segment, OverlayRelations.DERIVED_FROM)

In [ ]:
#| export
@dataclass
class TranscriptNode:
    """One transcriber's text for one AudioRendition (per-transcriber variants at
    the coarse layer — cross-transcriber divergence lives here, C4/C14)."""
    rendition: str                # Owning AudioRendition node id
    transcriber: str              # Transcriber capability name
    config_hash: str              # Transcriber config hash
    text: str                     # The transcript text (stored ONCE here; fine Segments slice into it)
    audio_hash: str               # Content hash of the consumed model-input WAV (the rendition's)
    metadata: Dict[str, Any] = field(default_factory=dict)  # Transcriber-reported metadata
    asserted_at: Optional[float] = None  # Derivation timestamp; None = now

    @property
    def id(self) -> str:  # Deterministic node id
        """Deterministic node id."""
        return transcript_node_id(self.rendition, self.transcriber, self.config_hash)

    def to_graph_node(self) -> Dict[str, Any]:  # Node wire dict
        """Build the Transcript node wire dict (capability attribution included)."""
        props: Dict[str, Any] = {
            "transcriber": self.transcriber,
            "config_hash": self.config_hash,
            "text": self.text,
            "rendition_id": self.rendition,
        }
        if self.metadata:
            props["metadata"] = dict(self.metadata)
        props.update(attribution(f"capability:{self.transcriber}", method="transcribe",
                                 asserted_at=self.asserted_at))
        return {
            "id": self.id,
            "label": TranscriptGraphLabels.TRANSCRIPT,
            "properties": props,
            "sources": [SourceRef(locator=GraphNodeRef(node_id=self.rendition),
                                  content_hash=self.audio_hash,
                                  slice=FullContent("audio")).to_dict()],
        }

    def derived_edge(self) -> Dict[str, Any]:  # Edge wire dict
        """DERIVED_FROM edge: this Transcript derives from its AudioRendition."""
        return make_edge(self.id, self.rendition, OverlayRelations.DERIVED_FROM)

In [ ]:
#| export
@dataclass
class TranscriptSliceRef:
    """One per-transcriber char-range reference for a fine Segment: where this
    segment's text lives inside a Transcript node's text (Thread-1
    slices-until-promoted — variants are slices, never duplicated fine text)."""
    transcript: str  # Transcript node id
    start_char: int  # Slice start into the Transcript's text
    end_char: int    # Slice end
    text: str        # The sliced text (content-hashed for the ref)

    def to_source_ref(self) -> Dict[str, Any]:  # SourceRef wire dict
        """Build the slice-shaped provenance ref."""
        return SourceRef(locator=GraphNodeRef(node_id=self.transcript),
                         content_hash=SourceRef.compute_hash(self.text.encode()),
                         slice=CharSlice(start=self.start_char, end=self.end_char)).to_dict()

In [ ]:
#| export
@dataclass
class SegmentNode:
    """Fine spine member: one VAD chunk — IMMUTABLE audio range + CORRECTABLE
    text (the immutable-audio/mutable-text spine seam). PART_OF its AudioRendition
    (each rendition has its own fine spine).

    Layer-0 `text` is the ACCURACY transcriber's alignment; the designation is
    per-segment provenance, not global config (`text_from` names the
    authoritative Transcript; every transcriber's char range rides
    `text_slices`, the authoritative one included)."""
    rendition: str            # Owning AudioRendition node id
    vad_config_hash: str      # VAD config hash (skeleton identity input)
    chunk_start: float        # VAD chunk start (chunk-local seconds within the rendition WAV)
    chunk_end: float          # VAD chunk end (chunk-local seconds)
    index: int                # Source-wide fine-spine index (0-based)
    start_time: float         # Source-coordinate start (navigation)
    end_time: float           # Source-coordinate end
    text: str = ""            # Layer-0 text (accuracy alignment; "" = no aligned words, D14 class)
    audio_hash: str = ""      # Content hash of the owning rendition's model-input WAV
    source: Optional[str] = None     # Source node id (convenience for direct filters)
    text_from: Optional[str] = None  # Authoritative Transcript node id (None when text is empty)
    text_slices: List[TranscriptSliceRef] = field(default_factory=list)  # All per-transcriber slice refs

    @property
    def id(self) -> str:  # Deterministic node id
        """Deterministic node id (audio-side identity; shared across transcribers, per-rendition)."""
        return segment_node_id(self.rendition, self.vad_config_hash,
                               self.chunk_start, self.chunk_end)

    def to_graph_node(self) -> Dict[str, Any]:  # Node wire dict
        """Build the Segment node wire dict (audio ref + per-transcriber text slice refs)."""
        props: Dict[str, Any] = {
            "text": self.text,
            "index": self.index,
            "start_time": self.start_time,
            "end_time": self.end_time,
            "chunk_start": self.chunk_start,
            "chunk_end": self.chunk_end,
            "rendition_id": self.rendition,
        }
        if self.source:
            props["source_id"] = self.source
        if self.text_from:
            props["text_from"] = self.text_from
        sources = [SourceRef(locator=GraphNodeRef(node_id=self.rendition),
                             content_hash=self.audio_hash,
                             slice=TimeSlice(start=self.chunk_start, end=self.chunk_end)).to_dict()]
        sources.extend(ts.to_source_ref() for ts in self.text_slices)
        return {
            "id": self.id,
            "label": TranscriptGraphLabels.SEGMENT,
            "properties": props,
            "sources": sources,
        }

In [ ]:
# tests — identity determinism + cross-type distinctness
sid = source_node_id("sha256:abc")
assert sid == source_node_id("sha256:abc")
a1 = audio_segment_node_id(sid, 0.0, 300.2)
assert a1 == audio_segment_node_id(sid, 0.0, 300.2)
assert a1 != audio_segment_node_id(sid, 0.0, 300.3)
# renditions: empty chain (raw) is stable + distinct from any preprocessing chain
r_raw = audio_rendition_node_id(a1, [])
assert r_raw == audio_rendition_node_id(a1, [])
r_vox = audio_rendition_node_id(a1, ["source_separation:demucs@cfg"])
assert r_vox != r_raw, "vocals rendition is a distinct node from raw"
assert r_raw != a1, "the raw rendition is its own node, not the audio segment"
assert audio_rendition_node_id(a1, ["a", "b"]) != audio_rendition_node_id(a1, ["b", "a"]), "chain order matters"
# transcript / segment key on the RENDITION now
t1 = transcript_node_id(r_raw, "whisper", "cfg1")
assert t1 != transcript_node_id(r_raw, "voxtral", "cfg1")
assert t1 != transcript_node_id(r_raw, "whisper", "cfg2")
assert t1 != transcript_node_id(r_vox, "whisper", "cfg1"), "raw vs vocals transcript distinct"
s1 = segment_node_id(r_raw, "vadcfg", 1.5, 4.25)
assert s1 == segment_node_id(r_raw, "vadcfg", 1.5, 4.25)
assert s1 != segment_node_id(r_vox, "vadcfg", 1.5, 4.25), "raw vs vocals fine spine distinct"
ids = {sid, a1, r_raw, r_vox, t1, s1}
assert len(ids) == 6, "kinds never collide"
print("identity tests OK")

In [ ]:
# tests — Source / AudioSegment / AudioRendition wire shapes
src = SourceNode(content_hash="sha256:abc", path="/media/ep1.mp3")
n = src.to_graph_node()
assert n["label"] == "Source" and n["id"] == source_node_id("sha256:abc")
assert n["properties"]["title"] == "ep1" and n["properties"]["root_kind"] == "ingested"
assert n["sources"][0]["content_hash"] == "sha256:abc"
assert SourceNode(content_hash="sha256:abc", path="/x.mp3", title="Custom").to_graph_node()["properties"]["title"] == "Custom"

# AudioSegment is a hashless boundary now (model-input moved to the rendition)
aseg = AudioSegmentNode(source=src.id, index=0, start=0.0, end=300.2, segment_path="/cuts/seg0.mp3")
an = aseg.to_graph_node()
assert an["id"] == audio_segment_node_id(src.id, 0.0, 300.2)
assert an["label"] == "AudioSegment"
assert an["properties"]["source_id"] == src.id and an["properties"]["segment_path"] == "/cuts/seg0.mp3"
assert "model_input_path" not in an["properties"], "model-input lives on the rendition now"
assert an["sources"] == [], "boundary node is hashless"

# AudioRendition owns the model-input; raw (empty chain) vs vocals coexist under one AudioSegment
raw = AudioRenditionNode(audio_segment=aseg.id, model_input_path="/cache/seg0.wav", model_input_hash="sha256:wav0")
rn = raw.to_graph_node()
assert rn["id"] == audio_rendition_node_id(aseg.id, []) and rn["label"] == "AudioRendition"
assert rn["properties"]["audio_segment_id"] == aseg.id and rn["properties"]["is_raw"] is True
assert rn["properties"]["chain"] == [] and "preprocessing" not in rn["properties"]
assert rn["sources"][0]["content_hash"] == "sha256:wav0"
re = raw.derived_edge()
assert re["relation_type"] == "DERIVED_FROM" and re["source_id"] == rn["id"] and re["target_id"] == aseg.id

vox = AudioRenditionNode(audio_segment=aseg.id, model_input_path="/cache/seg0_vocals.wav",
                         model_input_hash="sha256:wav0vox", chain=["source_separation:demucs@cfg"])
vn = vox.to_graph_node()
assert vn["id"] != rn["id"], "vocals rendition coexists as a distinct node"
assert vn["properties"]["is_raw"] is False and vn["properties"]["preprocessing"] == "source_separation:demucs@cfg"
assert vn["sources"][0]["content_hash"] == "sha256:wav0vox"
print("source/audio-segment/rendition shape tests OK")

In [ ]:
# tests — Transcript shape + attribution + derived edge (keyed on the rendition)
t = TranscriptNode(rendition=r_raw, transcriber="whisper", config_hash="cfg1",
                   text="hello world", audio_hash="sha256:wav0",
                   metadata={"model": "base"}, asserted_at=123.0)
tn = t.to_graph_node()
assert tn["id"] == transcript_node_id(r_raw, "whisper", "cfg1")
assert tn["properties"]["actor"] == "capability:whisper"
assert tn["properties"]["method"] == "transcribe" and tn["properties"]["asserted_at"] == 123.0
assert tn["properties"]["text"] == "hello world" and tn["properties"]["metadata"] == {"model": "base"}
assert tn["properties"]["rendition_id"] == r_raw
assert tn["sources"][0]["slice"]["kind"] == "full"
e = t.derived_edge()
assert e["relation_type"] == "DERIVED_FROM" and e["source_id"] == tn["id"] and e["target_id"] == r_raw
assert e["id"] == t.derived_edge()["id"], "deterministic edge id"
print("transcript shape tests OK")

In [ ]:
# tests — Segment shape: audio ref + authoritative/variant text slices; empty segment
acc = TranscriptSliceRef(transcript="t-acc", start_char=0, end_char=11, text="hello world")
var = TranscriptSliceRef(transcript="t-lw", start_char=0, end_char=10, text="helo world")
seg = SegmentNode(rendition=r_raw, vad_config_hash="vadcfg", chunk_start=1.5, chunk_end=4.25,
                  index=7, start_time=101.5, end_time=104.25, text="hello world",
                  audio_hash="sha256:wav0", source=sid, text_from="t-acc",
                  text_slices=[acc, var])
sn = seg.to_graph_node()
assert sn["id"] == segment_node_id(r_raw, "vadcfg", 1.5, 4.25)
assert sn["properties"]["text"] == "hello world" and sn["properties"]["text_from"] == "t-acc"
assert sn["properties"]["source_id"] == sid and sn["properties"]["rendition_id"] == r_raw
assert len(sn["sources"]) == 3
audio_ref, acc_ref, var_ref = sn["sources"]
assert audio_ref["slice"]["kind"] == "time" and audio_ref["slice"]["start"] == 1.5
assert audio_ref["locator"]["node_id"] == r_raw, "audio ref points at the rendition"
assert acc_ref["slice"]["kind"] == "char" and acc_ref["content_hash"] == SourceRef.compute_hash(b"hello world")
assert var_ref["content_hash"] == SourceRef.compute_hash(b"helo world")
# identity is audio-side: text changes do NOT change the id (skeleton shared across transcribers)
seg2 = SegmentNode(rendition=r_raw, vad_config_hash="vadcfg", chunk_start=1.5, chunk_end=4.25,
                   index=7, start_time=101.5, end_time=104.25, text="different text")
assert seg2.id == seg.id
# empty (D14-class) segment: audio ref only, no text_from
empty = SegmentNode(rendition=r_raw, vad_config_hash="vadcfg", chunk_start=9.0, chunk_end=9.5,
                    index=8, start_time=109.0, end_time=109.5, audio_hash="sha256:wav0")
en = empty.to_graph_node()
assert en["properties"]["text"] == "" and "text_from" not in en["properties"]
assert len(en["sources"]) == 1
print("segment shape tests OK")